# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [205]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [206]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = '../data/Malicious_URL_v3.csv'
# DATA_PATH = '../data/malicious_phish.csv' 

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 49750 rows x 3 columns


,Unnamed: 0,url,type
0,0,br-icloud.com.br,phishing
1,1,mp3raid.com/music/krizz_kaliko.html,benign
2,2,bopsecrets.org/rexroth/cr/1.htm,benign
3,3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,4,http://adventure-nicaragua.net/index.php?optio...,defacement


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [207]:
# TODO: Select the relevant columns and rows for your analysis.
# Document your rationale for including or excluding specific features.

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from src.clean import load_title_url

# Example:
columns_to_keep = ['url', 'type']
columns_to_drop = [df.columns[0]] 
drop_reason = {
    'df.columns[0]': 'Unique identifier, not predictive',
}

df_selected = df.drop(columns=columns_to_drop)
print(f"Shape after column selection: {df_selected.shape}")
df_selected.head()

# more data 
df_malware_suplement = load_title_url('malware.csv', 'malware')
df_phishing_suplement = load_title_url('phising.csv', 'phishing')

df_suplement = pd.concat([df_malware_suplement, df_phishing_suplement, df_selected], ignore_index=True)
print(f"Loaded suplement dataset: {df_suplement.shape[0]} rows x {df_suplement.shape[1]} columns")
df_suplement.head()

Shape after column selection: (49750, 2)
Loaded suplement dataset: 176107 rows x 2 columns


,url,type
0,http://31.208.67.180:33161/i,malware
1,https://def-system.fighttrapper.in.net/verific...,malware
2,http://31.208.67.180:33161/bin.sh,malware
3,https://target-api.fighttrapper.in.net/verific...,malware
4,http://59.184.247.189:54682/i,malware


In [208]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

# df_selected = df_selected[df_selected['some_column'].notna()]
# print(f"Shape after row selection: {df_selected.shape}")

---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [209]:
# TODO: Handle missing values.
# Choose an appropriate strategy for each column.

#df_clean = df_selected.copy()
df_clean = df_suplement.copy()

# 1. Handle missing values (minimal since dataset is mostly clean)
df_clean = df_clean.dropna(subset=['url', 'type'])

# 2. Remove empty URLs
df_clean = df_clean[df_clean['url'].str.strip() != ""]

# 3. Standardize target labels
df_clean['type'] = df_clean['type'].str.lower().str.strip()

# 4. Keep only valid classes
valid_classes = ['benign', 'phishing', 'malware', 'defacement']
df_clean = df_clean[df_clean['type'].isin(valid_classes)]

print(f"Shape after cleaning: {df_clean.shape}")
df_clean['type'].value_counts()
print(len(df_selected) - len(df_clean), "rows removed during cleaning.")



Shape after cleaning: (176107, 2)
-126357 rows removed during cleaning.


In [210]:
# TODO: Remove duplicate records.

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")



Removed 9318 duplicate rows. Remaining: 166789 rows.


In [211]:
# TODO: Handle outliers.
# Choose a strategy: capping (winsorizing), removing, or transforming.

# Example: Cap outliers using the IQR method
# def cap_outliers_iqr(dataframe, column):
#     Q1 = dataframe[column].quantile(0.25)
#     Q3 = dataframe[column].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR
#     dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
#     return dataframe

# for col in numerical_cols:
#     df_clean = cap_outliers_iqr(df_clean, col)

In [212]:
print("=== Outlier Handling ===\n")

print("At this stage, the dataset does not contain engineered numerical features suitable for traditional outlier treatment methods.\n")

print("A simple proxy variable (URL length) was analyzed during the Data Understanding phase, revealing the presence of extreme values.\n")

print("However, in the context of cybersecurity:")
print("- Outliers may represent malicious behavior (e.g., obfuscated URLs)")
print("- Removing or capping these values could eliminate important signals\n")

print("Decision:")
print("No outlier removal or transformation was applied.\n")

print("Justification:")
print("Outliers are retained as they are likely to carry meaningful information for detecting phishing and malicious URLs.\n")

print("Conclusion:")
print("The dataset will be further processed during feature engineering, where more meaningful numerical features will be created.")

=== Outlier Handling ===

At this stage, the dataset does not contain engineered numerical features suitable for traditional outlier treatment methods.

A simple proxy variable (URL length) was analyzed during the Data Understanding phase, revealing the presence of extreme values.

However, in the context of cybersecurity:
- Outliers may represent malicious behavior (e.g., obfuscated URLs)
- Removing or capping these values could eliminate important signals

Decision:
No outlier removal or transformation was applied.

Justification:
Outliers are retained as they are likely to carry meaningful information for detecting phishing and malicious URLs.

Conclusion:
The dataset will be further processed during feature engineering, where more meaningful numerical features will be created.


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [213]:
# TODO: Create derived attributes / new features.

from src.feature_extraction import extract_features
# Apply feature extraction

features_df = df_clean['url'].apply(extract_features).apply(pd.Series)

# Combine with target
df_features = pd.concat([features_df, df_clean['type']], axis=1)

print("Feature engineering completed.")
print(df_features.head())

Feature engineering completed.
   url_length  domain_length  path_length  num_dots  num_subdomains  \
0        28.0           19.0          2.0       3.0             2.0   
1        58.0           30.0         20.0       3.0             2.0   
2        33.0           19.0          7.0       3.0             2.0   
3        58.0           30.0         20.0       3.0             2.0   
4        29.0           20.0          2.0       3.0             2.0   

   num_digits  has_dash  has_ip  has_at  has_suspicious_words  \
0        15.0       0.0     1.0     0.0                   0.0   
1         0.0       1.0     0.0     0.0                   0.0   
2        15.0       0.0     1.0     0.0                   0.0   
3         0.0       1.0     0.0     0.0                   0.0   
4        16.0       0.0     1.0     0.0                   0.0   

   is_suspicious_tld  url_entropy     type  
0                0.0     3.655046  malware  
1                0.0     4.188843  malware  
2               

In [214]:
# TODO: Encode categorical variables.

# Example: One-hot encoding
# df_clean = pd.get_dummies(df_clean, columns=['categorical_col_1', 'categorical_col_2'], drop_first=True)

# Example: Label encoding for ordinal variables
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df_clean['ordinal_col'] = le.fit_transform(df_clean['ordinal_col'])

from sklearn.preprocessing import LabelEncoder

# Encode target variable
le = LabelEncoder()
df_features['type_encoded'] = le.fit_transform(df_features['type'])

print("Encoded classes:", list(le.classes_))
for i, cls in enumerate(le.classes_):
    print(f"{cls} -> {i}")


#since this is a target we can use label encoding
# if it was a feature we would use one hot encoding to avoid implying an order between the classes

Encoded classes: ['benign', 'defacement', 'malware', 'phishing']
benign -> 0
defacement -> 1
malware -> 2
phishing -> 3


In [215]:
# TODO: Scale / normalise numerical features if required.

# from sklearn.preprocessing import StandardScaler, MinMaxScaler

# scaler = StandardScaler()  # or MinMaxScaler()
# cols_to_scale = ['feature_1', 'feature_2']
# df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])

from sklearn.preprocessing import StandardScaler

df_features.head()

binary_cols = ['has_ip', 'has_https', 'has_dash']
numerical_cols = [col for col in df_features.columns 
                  if col not in ['type', 'type_encoded'] + binary_cols]

scaler = StandardScaler()
df_features[numerical_cols] = scaler.fit_transform(df_features[numerical_cols])
df_features.head()

,url_length,domain_length,path_length,num_dots,num_subdomains,num_digits,has_dash,has_ip,has_at,has_suspicious_words,is_suspicious_tld,url_entropy,type,type_encoded
0,-0.251864,0.313105,-0.750747,1.01454,1.107624,0.405464,0.0,1.0,-0.054115,-0.146298,-0.073161,-1.186513,malware,2
1,0.049320,1.299021,-0.143666,1.01454,1.107624,-0.542261,1.0,0.0,-0.054115,-0.146298,-0.073161,-0.032322,malware,2
2,-0.201667,0.313105,-0.582113,1.01454,1.107624,0.405464,0.0,1.0,-0.054115,-0.146298,-0.073161,-0.640143,malware,2
3,0.049320,1.299021,-0.143666,1.01454,1.107624,-0.542261,1.0,0.0,-0.054115,-0.146298,-0.073161,-0.501081,malware,2
4,-0.241825,0.402733,-0.750747,1.01454,1.107624,0.468645,0.0,1.0,-0.054115,-0.146298,-0.073161,-0.898297,malware,2


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [216]:
# TODO: Integrate data from multiple sources (if applicable).

# i integrated before feature engineering


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [217]:
# TODO: Apply final formatting — data types, column order, renaming.

df_final = df_features.copy()

# Ensure correct data types
df_final['type'] = df_final['type'].astype('category')
df_final['type_encoded'] = df_final['type_encoded'].astype('int')

# Reorder columns (features first, target last)
target_col = 'type_encoded'
feature_cols = [col for col in df_final.columns if col not in ['type', 'type_encoded']]

df_final = df_final[feature_cols + ['type', 'type_encoded']]

print("Final formatting applied.")

Final formatting applied.


In [218]:
# TODO: Verify the final prepared dataset.

print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)

print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicates: {df_final.duplicated().sum()}")

print("\nColumn types:")
print(df_final.dtypes)

print("\nPreview:")
print(df_final.head())

FINAL PREPARED DATASET SUMMARY
Shape: (166789, 14)
Missing values: 0
Duplicates: 72675

Column types:
url_length               float64
domain_length            float64
path_length              float64
num_dots                 float64
num_subdomains           float64
num_digits               float64
has_dash                 float64
has_ip                   float64
has_at                   float64
has_suspicious_words     float64
is_suspicious_tld        float64
url_entropy              float64
type                    category
type_encoded               int64
dtype: object

Preview:
   url_length  domain_length  path_length  num_dots  num_subdomains  \
0   -0.251864       0.313105    -0.750747   1.01454        1.107624   
1    0.049320       1.299021    -0.143666   1.01454        1.107624   
2   -0.201667       0.313105    -0.582113   1.01454        1.107624   
3    0.049320       1.299021    -0.143666   1.01454        1.107624   
4   -0.241825       0.402733    -0.750747   1.01454      

In [219]:
print("="*40)
print("DUPLICATE HANDLING")
print("="*40)

print("A significant number of duplicate rows were identified after feature engineering.")
print("These duplicates arise because different URLs can produce identical feature values.\n")

print("Action Taken:")
print("Duplicate rows were removed to prevent bias and improve model generalization.\n")

print("Conclusion:")
print("The dataset now contains only unique feature representations, making it more suitable for training.")

DUPLICATE HANDLING
A significant number of duplicate rows were identified after feature engineering.
These duplicates arise because different URLs can produce identical feature values.

Action Taken:
Duplicate rows were removed to prevent bias and improve model generalization.

Conclusion:
The dataset now contains only unique feature representations, making it more suitable for training.


In [220]:
df_final = df_final.drop_duplicates()

print("After removing duplicates:")
print(f"Shape: {df_final.shape}")
print(f"Duplicates: {df_final.duplicated().sum()}")

counts = df_final['type'].value_counts()
percentages = df_final['type'].value_counts(normalize=True) * 100

result = df_final['type'].value_counts().to_frame(name='count')
result['percentage'] = percentages

print(result)

After removing duplicates:
Shape: (94114, 14)
Duplicates: 0
            count  percentage
type                         
phishing    35900   38.145228
malware     30215   32.104682
benign      21023   22.337803
defacement   6976    7.412287


In [221]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

OUTPUT_PATH = "../data/malicious_urls_prepared_supplement.csv"

df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Prepared dataset saved to: {OUTPUT_PATH}")

Prepared dataset saved to: ../data/malicious_urls_prepared_supplement.csv
